# 中证800 V65：滚动训练方法论验证（训练长度与 cutoff 效应）

这个 notebook 回答一个比“哪个固定模型最好”更重要的问题：

> 如果未来真实运行这套方法，每隔一段时间滚动重训，它是否能持续产生 OOS 超额？

本实验固定当前主线方法，不继续发明新模型：

- full 特征集
- V46/V61 LightGBM 参数
- fixed 120 rounds
- legacy_unsealed 训练边界
- `top8_board_cap` 组合构建：8 只，创业板最多 3 只，科创板最多 2 只
- 每 6 个月重训一次，交易未来 6 个月

核心比较：

1. **训练长度效应**：rolling36 / rolling48 / rolling60 / rolling72 / expanding 谁更稳？
2. **cutoff/regime 效应**：某些 cutoff 之后好，是因为数据量更大，还是因为训练集包含了关键市场状态？
3. **生产可运行性**：把每个训练方法拼成连续 OOS 曲线，而不是事后挑最好 cutoff。

输出既有 proxy OOS，也有可选 JQ-like 日频账户模拟：如果在聚宽环境能调用 `get_price`，会自动跑更接近真实回测的账户路径；如果本地普通 Python 环境不可用，则自动跳过。


In [ ]:
import os
import gc
import warnings
from pathlib import Path
from typing import Dict, List, Any, Optional

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()

# =========================
# Config
# =========================
DATA_PATH = "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"
OUT_DIR = Path("csi800_ml_v65_rolling_retrain_methodology_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"
BENCHMARK = "000906.XSHG"
MODEL_FAMILY = "v65_rolling_retrain_methodology_validation"
BOUNDARY_POLICY = "legacy_unsealed_q4"
USE_LEGACY_UNSEALED_BOUNDARY = True

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70

PORTFOLIO_RULE = "top8_board_cap"
STOCK_NUM = 8
BOARD_CAPS = {"chinext": 3, "star": 2}
BOARD_CAPS_TEXT = ";".join(["%s:%s" % (k, BOARD_CAPS[k]) for k in sorted(BOARD_CAPS)])

# 近似交易成本，与 V61 保持一致。proxy OOS 用它做初步净值；JQ-like 账户会再用同一套成本模拟现金路径。
SLIPPAGE_RATE = 0.00246
OPEN_COMMISSION = 0.0003
CLOSE_COMMISSION = 0.0003
CLOSE_TAX = 0.001
BUY_COST_RATE = SLIPPAGE_RATE + OPEN_COMMISSION
SELL_COST_RATE = SLIPPAGE_RATE + CLOSE_COMMISSION + CLOSE_TAX

RANDOM_SIM_N = 500
RANDOM_SEED = 42

RETRAIN_EVERY_MONTHS = 6
OOS_MONTHS = 6
MIN_OOS_MONTHS = 3
TRAIN_POLICIES = [
    {"train_policy": "expanding_min36", "train_window_months": None, "min_train_months": 36, "method_type": "expanding"},
    {"train_policy": "rolling36m", "train_window_months": 36, "min_train_months": 36, "method_type": "rolling"},
    {"train_policy": "rolling48m", "train_window_months": 48, "min_train_months": 48, "method_type": "rolling"},
    {"train_policy": "rolling60m", "train_window_months": 60, "min_train_months": 60, "method_type": "rolling"},
    {"train_policy": "rolling72m", "train_window_months": 72, "min_train_months": 72, "method_type": "rolling"},
]
CORE_COMMON_POLICIES = ["expanding_min36", "rolling36m", "rolling48m", "rolling60m"]

# 门槛只用于方法论健康判断，不用于调参。
PASS_MIN_MONTHS = 24
PASS_MIN_CUM_RET = 0.0
PASS_MIN_WIN_RATE = 0.55
PASS_MIN_POSITIVE_FOLD_RATE = 0.60
PASS_MIN_RANDOM_PERCENTILE = 0.55
PASS_MAX_DRAWDOWN = -0.25

# 聚宽环境下可自动跑；本地无 get_price 时自动跳过。
RUN_JQ_LIKE_SIMULATOR = "AUTO"
JQ_SIM_INITIAL_CASH = 300000.0
JQ_SIM_PRICE_FQ = None
JQ_SIM_BUY_TIME_FIELD = "open"
JQ_SIM_MARK_FIELD = "close"
JQ_SIM_CHUNK_SIZE = 120
MIN_COMMISSION = 5.0
NORMAL_MIN_LOT = 100
KCB_MIN_LOT = 200

print("DATA_PATH:", DATA_PATH)
print("OUT_DIR:", OUT_DIR.resolve())
print("portfolio:", PORTFOLIO_RULE, "stock_num", STOCK_NUM, "board_caps", BOARD_CAPS_TEXT)
print("train policies:", [x["train_policy"] for x in TRAIN_POLICIES])


In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio",
    "book_to_price_ratio",
    "earnings_yield",
    "sales_to_price_ratio",
    "cash_earnings_to_price_ratio",
    "earnings_to_price_ratio",
    "roe_ttm",
    "roa_ttm",
    "gross_profit_ttm",
    "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit",
    "ACCA",
    "growth",
    "net_working_capital",
    "operating_profit_per_share",
    "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share",
    "super_quick_ratio",
    "MLEV",
    "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio",
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "liquidity",
    "beta",
    "ATR6",
    "MFI14",
    "DAVOL10",
    "VOL10",
    "VMACD",
    "VOSC",
    "Skewness20",
    "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

FULL_FEATURE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 1.0,
    "bagging_freq": 0,
    "lambda_l1": 0.0,
    "lambda_l2": 0.0,
    "verbosity": -1,
    "num_threads": 4,
}


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def calc_nav(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return (1.0 + s).cumprod() if len(s) else pd.Series(dtype=float)


def calc_drawdown_from_returns(ret_series):
    nav = calc_nav(ret_series)
    if len(nav) == 0:
        return np.nan
    return float((nav / nav.cummax() - 1.0).min())


def summarize_return_series(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"months": 0, "cum_ret": np.nan, "ann_ret": np.nan, "mean_ret": np.nan, "win_rate": np.nan, "max_drawdown": np.nan, "sharpe12": np.nan}
    std = s.std(ddof=1)
    return {
        "months": int(len(s)),
        "cum_ret": float((1.0 + s).prod() - 1.0),
        "ann_ret": float((1.0 + s).prod() ** (12.0 / len(s)) - 1.0) if len(s) else np.nan,
        "mean_ret": float(s.mean()),
        "win_rate": float((s > 0).mean()),
        "max_drawdown": calc_drawdown_from_returns(s),
        "sharpe12": float(s.mean() / std * np.sqrt(12)) if len(s) > 1 and std > 0 else np.nan,
    }


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []
    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)
    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    usable = []
    for col in unique_keep_order(candidate_cols):
        if col not in train_df.columns:
            continue
        s = pd.to_numeric(train_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        if s.notnull().sum() < max(20, int(len(train_df) * 0.05)):
            continue
        if s.nunique(dropna=True) < 2:
            continue
        usable.append(col)
    if len(usable) == 0:
        raise ValueError("no usable features")
    comps = build_corr_components(train_df, usable, CORR_THRESHOLD)
    ic_map = {}
    for col in usable:
        ic_map[col] = abs(safe_rank_ic(train_df[col], train_df[TARGET_COL]))
    kept = []
    removed = []
    for comp in comps:
        if len(comp) == 1:
            kept.append(comp[0])
            continue
        comp_sorted = sorted(comp, key=lambda x: (ic_map.get(x, 0.0), -usable.index(x)), reverse=True)
        kept.append(comp_sorted[0])
        removed.extend(comp_sorted[1:])
    return unique_keep_order(kept), removed


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    if fill_values is None:
        fill_values = X.median(numeric_only=True).to_dict()
    X = X.fillna(fill_values).fillna(0)
    y = pd.to_numeric(d[target_col], errors="coerce").astype(float)
    ok = y.notnull().values
    return X.loc[ok, feature_cols], y.loc[ok], fill_values, d.loc[ok].index


def split_diag_valid(train_df, valid_months=6):
    months = sorted(pd.to_datetime(train_df[DATE_COL].dropna().unique()))
    if len(months) <= valid_months + 6:
        return train_df.copy(), train_df.copy()
    valid_set = set(months[-valid_months:])
    fit = train_df[~train_df[DATE_COL].isin(valid_set)].copy()
    valid = train_df[train_df[DATE_COL].isin(valid_set)].copy()
    return fit, valid


def stable_text_seed(text):
    total = 0
    for i, ch in enumerate(str(text)):
        total += (i + 1) * ord(ch)
    return int(total % 100000)


In [ ]:
def first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None


def load_dataset(path):
    if not os.path.exists(path):
        raise IOError("DATA_PATH not found: " + path)
    df = pd.read_csv(path)
    if "code" in df.columns and STOCK_COL not in df.columns:
        df = df.rename(columns={"code": STOCK_COL})
    for col in [DATE_COL, "feature_date", "next_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce").dt.normalize()
    if STOCK_COL not in df.columns:
        raise ValueError("missing stock column")
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = df["raw_return_1m"] - df["benchmark_csi800_1m"]
        else:
            raise ValueError("missing target: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[STOCK_COL, DATE_COL, TARGET_COL]).copy()
    return df


def infer_return_columns(df):
    raw_col = first_existing(df.columns, ["raw_return_1m", "stock_return_1m", "return_1m", "next_return_1m"])
    bench_col = first_existing(df.columns, ["benchmark_csi800_1m", "benchmark_000906_1m", "benchmark_alla_1m", "cum_csi800_1m"])
    alpha_col = TARGET_COL if TARGET_COL in df.columns else None
    if raw_col is None and alpha_col is not None and bench_col is not None:
        df["_v65_raw_return_1m"] = df[alpha_col] + df[bench_col]
        raw_col = "_v65_raw_return_1m"
    return raw_col, bench_col, alpha_col


def get_month_benchmark_return(month_df):
    if BENCH_RET_COL is None:
        return np.nan
    s = pd.to_numeric(month_df[BENCH_RET_COL], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    return float(s.iloc[0]) if len(s) else np.nan


def get_return_map(month_df, col):
    if col is None:
        return {}
    return dict(zip(month_df[STOCK_COL].astype(str), pd.to_numeric(month_df[col], errors="coerce")))


def calc_portfolio_return_from_map(ret_map, stocks):
    vals = []
    for stock in stocks:
        v = ret_map.get(stock, np.nan)
        if not pd.isnull(v):
            vals.append(float(v))
    return float(np.mean(vals)) if vals else np.nan


df_all = load_dataset(DATA_PATH)
RAW_RET_COL, BENCH_RET_COL, ALPHA_RET_COL = infer_return_columns(df_all)
available_features = [c for c in FULL_FEATURE_COLS if c in df_all.columns]
missing_features = [c for c in FULL_FEATURE_COLS if c not in df_all.columns]
months_all = sorted(pd.to_datetime(df_all[DATE_COL].dropna().unique()))

print("loaded:", df_all.shape)
print("date range:", pd.Timestamp(months_all[0]).date(), "->", pd.Timestamp(months_all[-1]).date(), "months", len(months_all))
print("raw:", RAW_RET_COL, "bench:", BENCH_RET_COL, "alpha:", ALPHA_RET_COL)
print("features available/missing:", len(available_features), len(missing_features))
if missing_features:
    print("missing features:", ",".join(missing_features))
display(df_all[[TARGET_COL]].describe())


In [ ]:
def make_fold_plan_for_policy(months, policy):
    months = list(sorted(pd.to_datetime(months)))
    rows = []
    min_train_months = int(policy["min_train_months"])
    start_idx = min_train_months
    fold_no = 0
    while start_idx < len(months):
        test_slice = months[start_idx:start_idx + OOS_MONTHS]
        if len(test_slice) < MIN_OOS_MONTHS:
            break
        if policy.get("train_window_months") is None:
            train_slice = months[:start_idx]
        else:
            w = int(policy["train_window_months"])
            train_slice = months[max(0, start_idx - w):start_idx]
        if len(train_slice) < min_train_months:
            start_idx += RETRAIN_EVERY_MONTHS
            continue
        fold_no += 1
        rows.append({
            "train_policy": policy["train_policy"],
            "method_type": policy["method_type"],
            "train_window_months": policy.get("train_window_months"),
            "fold_id": "%s_%02d" % (policy["train_policy"], fold_no),
            "fold_no": fold_no,
            "train_start": train_slice[0],
            "train_end": train_slice[-1],
            "test_start": test_slice[0],
            "test_end": test_slice[-1],
            "train_months": len(train_slice),
            "test_months": len(test_slice),
            "retrain_every_months": RETRAIN_EVERY_MONTHS,
            "oos_months": OOS_MONTHS,
        })
        start_idx += RETRAIN_EVERY_MONTHS
    return rows


fold_plan_rows = []
for policy in TRAIN_POLICIES:
    fold_plan_rows.extend(make_fold_plan_for_policy(months_all, policy))
fold_plan_df = pd.DataFrame(fold_plan_rows)
if fold_plan_df.empty:
    raise ValueError("empty fold plan")

# Shared comparison windows, to avoid mistaking later-starting long windows for better/worse methods.
first_month_by_policy = fold_plan_df.groupby("train_policy")["test_start"].min()
common_all_start = first_month_by_policy.max()
core_available = [p for p in CORE_COMMON_POLICIES if p in first_month_by_policy.index]
common_core_start = first_month_by_policy.loc[core_available].max() if core_available else common_all_start
print("common_all_start:", pd.Timestamp(common_all_start).date())
print("common_core_start:", pd.Timestamp(common_core_start).date(), "core:", core_available)
display(fold_plan_df)
fold_plan_df.to_csv(OUT_DIR / "v65_fold_plan.csv", index=False)


In [ ]:
def board_type(stock):
    s = str(stock)
    if s.startswith("30"):
        return "chinext"
    if s.startswith(("688", "689")):
        return "star"
    return "main"


def board_cap_allows(selected, stock, board_caps):
    if not board_caps:
        return True
    b = board_type(stock)
    if b not in board_caps:
        return True
    return sum(1 for x in selected if board_type(x) == b) < int(board_caps[b])


def build_board_capped_targets(sorted_stocks, target_num=STOCK_NUM, board_caps=BOARD_CAPS):
    selected = []
    for stock in sorted_stocks:
        if stock in selected:
            continue
        if board_cap_allows(selected, stock, board_caps):
            selected.append(stock)
        if len(selected) >= target_num:
            return selected[:target_num]
    # Fallback: if caps make the target count impossible, fill by score without caps.
    for stock in sorted_stocks:
        if stock not in selected:
            selected.append(stock)
        if len(selected) >= target_num:
            break
    return selected[:target_num]


def summarize_target_board(targets):
    out = {"board_main": 0, "board_chinext": 0, "board_star": 0}
    for stock in targets:
        b = board_type(stock)
        out["board_" + b] = out.get("board_" + b, 0) + 1
    n = max(1, len(targets))
    out["board_main_ratio"] = out.get("board_main", 0) / float(n)
    out["board_chinext_ratio"] = out.get("board_chinext", 0) / float(n)
    out["board_star_ratio"] = out.get("board_star", 0) / float(n)
    out["board_hhi"] = sum((out[k] / float(n)) ** 2 for k in ["board_main", "board_chinext", "board_star"])
    return out


def calc_equal_weight_turnover(prev_targets, targets):
    if len(targets) == 0:
        return 0.0, 0.0, 0
    if prev_targets is None:
        return 1.0, 0.0, 0
    overlap = len(set(prev_targets).intersection(set(targets)))
    one_way = 1.0 - overlap / float(max(1, len(targets)))
    return float(one_way), float(one_way), int(overlap)


def calc_trade_cost(prev_targets, targets):
    buy_turnover, sell_turnover, overlap = calc_equal_weight_turnover(prev_targets, targets)
    cost = buy_turnover * BUY_COST_RATE + sell_turnover * SELL_COST_RATE
    if prev_targets is None:
        cost = buy_turnover * BUY_COST_RATE
    return float(cost), float(buy_turnover), float(sell_turnover), int(overlap)


def random_percentile_board_cap(month_df, actual_ret, ret_col, seed_key):
    if ret_col is None or pd.isnull(actual_ret):
        return np.nan
    d = month_df.dropna(subset=[ret_col]).copy().reset_index(drop=True)
    if d.empty:
        return np.nan
    stocks = d[STOCK_COL].astype(str).values
    rets = pd.to_numeric(d[ret_col], errors="coerce").replace([np.inf, -np.inf], np.nan).values.astype(float)
    valid = np.isfinite(rets)
    stocks = stocks[valid]
    rets = rets[valid]
    if len(rets) == 0:
        return np.nan
    stock_to_idx = {s: i for i, s in enumerate(stocks)}
    base_idx = np.arange(len(stocks), dtype=np.int32)
    rng = np.random.RandomState(seed_key)
    vals = np.empty(int(RANDOM_SIM_N), dtype=float)
    rand_iter = progress_iter(range(RANDOM_SIM_N), total=RANDOM_SIM_N, desc="random", leave=False)
    for i in rand_iter:
        perm = rng.permutation(base_idx)
        picked_stocks = build_board_capped_targets([stocks[j] for j in perm], STOCK_NUM, BOARD_CAPS)
        picked_idx = [stock_to_idx[s] for s in picked_stocks if s in stock_to_idx]
        vals[i] = float(np.nanmean(rets[picked_idx])) if len(picked_idx) else np.nan
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna()
    return float((vals <= actual_ret).mean()) if len(vals) else np.nan


In [ ]:
def train_policy_fold_model(train_df, fold_row):
    diag_fit_df, diag_valid_df = split_diag_valid(train_df)
    feature_cols, removed_cols = select_features_train_only(diag_fit_df, available_features)
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED + stable_text_seed(fold_row["fold_id"])
    X_train, y_train, fill_values, _ = prepare_xy(train_df, feature_cols, TARGET_COL)
    if len(X_train) == 0:
        raise ValueError("empty training matrix for " + str(fold_row["fold_id"]))
    model = lgb.train(params, lgb.Dataset(X_train, label=y_train), num_boost_round=max(1, int(FIXED_ITER)))
    train_pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    train_rank_ic = safe_rank_ic(y_train, train_pred)
    X_valid, y_valid, _, _ = prepare_xy(diag_valid_df, feature_cols, TARGET_COL, fill_values)
    if len(X_valid):
        valid_pred = np.asarray(model.predict(X_valid[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
        diag_rank_ic = safe_rank_ic(y_valid, valid_pred)
    else:
        diag_rank_ic = np.nan
    return {
        "model": model,
        "feature_cols": feature_cols,
        "removed_cols": removed_cols,
        "fill_values": fill_values,
        "train_rows": int(len(X_train)),
        "train_rank_ic": train_rank_ic,
        "diag_rank_ic": diag_rank_ic,
    }


def score_with_trained_model(test_df, trained):
    X = test_df.reindex(columns=trained["feature_cols"]).replace([np.inf, -np.inf], np.nan).copy()
    X = X.fillna(trained["fill_values"]).fillna(0)
    out = test_df.copy()
    out["score"] = np.asarray(trained["model"].predict(X[trained["feature_cols"]], num_iteration=FIXED_ITER)).reshape(-1)
    out["score_rank_pct"] = out.groupby(DATE_COL)["score"].rank(pct=True)
    out["realized_rank_pct"] = out.groupby(DATE_COL)[TARGET_COL].rank(pct=True)
    return out


def evaluate_fold(fold_row, prev_targets=None):
    train_df = df_all[(df_all[DATE_COL] >= fold_row["train_start"]) & (df_all[DATE_COL] <= fold_row["train_end"])].copy()
    if not USE_LEGACY_UNSEALED_BOUNDARY and "next_date" in train_df.columns:
        train_df = train_df[train_df["next_date"] <= fold_row["train_end"]].copy()
    test_df = df_all[(df_all[DATE_COL] >= fold_row["test_start"]) & (df_all[DATE_COL] <= fold_row["test_end"])].copy()
    if train_df.empty or test_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), {"prev_targets": prev_targets}

    trained = train_policy_fold_model(train_df, fold_row)
    scored = score_with_trained_model(test_df, trained)

    monthly_rows = []
    panel_parts = []
    month_groups = list(scored.groupby(DATE_COL))
    for dt, month_df in progress_iter(month_groups, total=len(month_groups), desc="months %s" % fold_row["fold_id"], leave=False):
        m = month_df.dropna(subset=[TARGET_COL, "score"]).copy()
        if m.empty:
            continue
        sorted_stocks = list(m.sort_values("score", ascending=False)[STOCK_COL].astype(str))
        targets = build_board_capped_targets(sorted_stocks, STOCK_NUM, BOARD_CAPS)
        cost, buy_turnover, sell_turnover, overlap = calc_trade_cost(prev_targets, targets)
        raw_ret_map = get_return_map(m, RAW_RET_COL)
        alpha_ret_map = get_return_map(m, ALPHA_RET_COL)
        gross_alpha_ret = calc_portfolio_return_from_map(alpha_ret_map, targets)
        gross_raw_ret = calc_portfolio_return_from_map(raw_ret_map, targets) if RAW_RET_COL is not None else np.nan
        benchmark_ret = get_month_benchmark_return(m)
        proxy_net_alpha_ret = gross_alpha_ret - cost if not pd.isnull(gross_alpha_ret) else np.nan
        if not pd.isnull(gross_raw_ret):
            proxy_net_raw_ret = (1.0 + gross_raw_ret) * (1.0 - cost) - 1.0
            proxy_net_excess_ret = proxy_net_raw_ret - benchmark_ret if not pd.isnull(benchmark_ret) else proxy_net_alpha_ret
        else:
            proxy_net_raw_ret = np.nan
            proxy_net_excess_ret = proxy_net_alpha_ret

        real_sorted = m.sort_values(TARGET_COL, ascending=False)
        actual_top10 = set(real_sorted.head(10)[STOCK_COL].astype(str))
        actual_top20 = set(real_sorted.head(20)[STOCK_COL].astype(str))
        selected_rows = m.set_index(STOCK_COL).reindex(targets)
        seed_key = int(pd.Timestamp(dt).strftime("%Y%m%d")) + stable_text_seed(fold_row["fold_id"])
        board = summarize_target_board(targets)
        row = {
            "train_policy": fold_row["train_policy"],
            "method_type": fold_row["method_type"],
            "train_window_months": fold_row.get("train_window_months"),
            "fold_id": fold_row["fold_id"],
            "fold_no": fold_row["fold_no"],
            "train_start": fold_row["train_start"],
            "train_end": fold_row["train_end"],
            "test_start": fold_row["test_start"],
            "test_end": fold_row["test_end"],
            DATE_COL: dt,
            "next_date": m["next_date"].iloc[0] if "next_date" in m.columns else pd.NaT,
            "portfolio_rule": PORTFOLIO_RULE,
            "stock_num": STOCK_NUM,
            "board_caps": BOARD_CAPS_TEXT,
            "target_count": len(targets),
            "gross_alpha_ret": gross_alpha_ret,
            "gross_raw_ret": gross_raw_ret,
            "benchmark_ret": benchmark_ret,
            "trade_cost": cost,
            "buy_turnover": buy_turnover,
            "sell_turnover": sell_turnover,
            "target_overlap_prev": overlap,
            "proxy_net_alpha_ret": proxy_net_alpha_ret,
            "proxy_net_raw_ret": proxy_net_raw_ret,
            "proxy_net_excess_ret": proxy_net_excess_ret,
            "rank_ic": safe_rank_ic(m["score"], m[TARGET_COL]),
            "target_avg_realized_rank": float(selected_rows["realized_rank_pct"].mean()),
            "target_worst_realized_rank": float(selected_rows["realized_rank_pct"].min()),
            "selected_hit_real_top10": len(set(targets) & actual_top10),
            "selected_hit_real_top20": len(set(targets) & actual_top20),
            "selected_hit_rate_real_top20": len(set(targets) & actual_top20) / float(max(1, len(targets))),
            "random_alpha_percentile": random_percentile_board_cap(m, gross_alpha_ret, ALPHA_RET_COL, seed_key),
            "random_raw_percentile": random_percentile_board_cap(m, gross_raw_ret, RAW_RET_COL, seed_key + 17) if RAW_RET_COL is not None else np.nan,
            "targets": ",".join(targets),
            "candidate_top30": ",".join(sorted_stocks[:min(30, len(sorted_stocks))]),
        }
        row.update(board)
        monthly_rows.append(row)
        m["train_policy"] = fold_row["train_policy"]
        m["fold_id"] = fold_row["fold_id"]
        panel_parts.append(m)
        prev_targets = list(targets)

    meta = dict(fold_row)
    meta.update({
        "train_rows": trained["train_rows"],
        "feature_count": len(trained["feature_cols"]),
        "removed_feature_count": len(trained["removed_cols"]),
        "train_rank_ic": trained["train_rank_ic"],
        "diag_rank_ic": trained["diag_rank_ic"],
        "feature_cols": ",".join(trained["feature_cols"]),
        "removed_features": ",".join(trained["removed_cols"]),
    })
    monthly_df = pd.DataFrame(monthly_rows)
    score_panel_df = pd.concat(panel_parts, ignore_index=True, sort=False) if panel_parts else pd.DataFrame()
    return monthly_df, pd.DataFrame([meta]), score_panel_df, {"prev_targets": prev_targets}


In [ ]:
monthly_parts = []
fold_meta_parts = []
score_panel_parts = []

for train_policy, plan in progress_iter(list(fold_plan_df.groupby("train_policy")), total=fold_plan_df["train_policy"].nunique(), desc="train policies"):
    prev_targets = None
    plan = plan.sort_values("test_start").copy()
    for _, fold_row in progress_iter(list(plan.iterrows()), total=len(plan), desc="folds %s" % train_policy):
        monthly_one, meta_one, panel_one, state = evaluate_fold(fold_row.to_dict(), prev_targets=prev_targets)
        prev_targets = state.get("prev_targets")
        if not monthly_one.empty:
            monthly_parts.append(monthly_one)
        if not meta_one.empty:
            fold_meta_parts.append(meta_one)
        if not panel_one.empty:
            score_panel_parts.append(panel_one)
        gc.collect()

rolling_monthly_df = pd.concat(monthly_parts, ignore_index=True, sort=False) if monthly_parts else pd.DataFrame()
fold_meta_df = pd.concat(fold_meta_parts, ignore_index=True, sort=False) if fold_meta_parts else pd.DataFrame()
score_panel_df = pd.concat(score_panel_parts, ignore_index=True, sort=False) if score_panel_parts else pd.DataFrame()

if rolling_monthly_df.empty:
    raise ValueError("empty rolling monthly result")
rolling_monthly_df = rolling_monthly_df.sort_values(["train_policy", DATE_COL]).reset_index(drop=True)
rolling_monthly_df["proxy_alpha_nav"] = rolling_monthly_df.groupby("train_policy")["proxy_net_alpha_ret"].transform(lambda s: calc_nav(s).values if len(s) else s)
rolling_monthly_df["proxy_excess_nav"] = rolling_monthly_df.groupby("train_policy")["proxy_net_excess_ret"].transform(lambda s: calc_nav(s).values if len(s) else s)
rolling_monthly_df["proxy_excess_drawdown"] = rolling_monthly_df.groupby("train_policy")["proxy_net_excess_ret"].transform(lambda s: (calc_nav(s) / calc_nav(s).cummax() - 1.0).values if len(s) else s)

print("monthly/fold_meta/score_panel:", rolling_monthly_df.shape, fold_meta_df.shape, score_panel_df.shape)
display(fold_meta_df.head())
display(rolling_monthly_df.tail(20))


In [ ]:
def summarize_policy_monthly(g, ret_col="proxy_net_excess_ret"):
    g = g.sort_values(DATE_COL).copy()
    st = summarize_return_series(g[ret_col])
    fold_rows = []
    for fold_id, fg in g.groupby("fold_id"):
        fst = summarize_return_series(fg.sort_values(DATE_COL)[ret_col])
        fold_rows.append({"fold_id": fold_id, "fold_cum_ret": fst["cum_ret"], "fold_months": fst["months"]})
    fold_df = pd.DataFrame(fold_rows)
    positive_fold_rate = float((fold_df["fold_cum_ret"] > 0).mean()) if len(fold_df) else np.nan
    robust_pass = bool(
        st["months"] >= PASS_MIN_MONTHS
        and st["cum_ret"] > PASS_MIN_CUM_RET
        and st["win_rate"] >= PASS_MIN_WIN_RATE
        and positive_fold_rate >= PASS_MIN_POSITIVE_FOLD_RATE
        and float(pd.to_numeric(g["random_alpha_percentile"], errors="coerce").mean()) >= PASS_MIN_RANDOM_PERCENTILE
        and st["max_drawdown"] >= PASS_MAX_DRAWDOWN
    )
    return pd.Series({
        "train_policy": g["train_policy"].iloc[0],
        "method_type": g["method_type"].iloc[0],
        "train_window_months": g["train_window_months"].dropna().iloc[0] if g["train_window_months"].notnull().any() else np.nan,
        "months": st["months"],
        "folds": int(g["fold_id"].nunique()),
        "start": g[DATE_COL].min(),
        "end": g[DATE_COL].max(),
        "ret_col": ret_col,
        "cum_ret": st["cum_ret"],
        "ann_ret": st["ann_ret"],
        "mean_monthly_ret": st["mean_ret"],
        "win_rate": st["win_rate"],
        "max_drawdown": st["max_drawdown"],
        "sharpe12": st["sharpe12"],
        "positive_fold_rate": positive_fold_rate,
        "worst_fold_cum_ret": float(fold_df["fold_cum_ret"].min()) if len(fold_df) else np.nan,
        "median_fold_cum_ret": float(fold_df["fold_cum_ret"].median()) if len(fold_df) else np.nan,
        "rank_ic_mean": float(pd.to_numeric(g["rank_ic"], errors="coerce").mean()),
        "rank_ic_positive_rate": float((pd.to_numeric(g["rank_ic"], errors="coerce") > 0).mean()),
        "avg_random_alpha_percentile": float(pd.to_numeric(g["random_alpha_percentile"], errors="coerce").mean()),
        "p25_random_alpha_percentile": float(pd.to_numeric(g["random_alpha_percentile"], errors="coerce").quantile(0.25)),
        "avg_selected_hit_rate_real_top20": float(pd.to_numeric(g["selected_hit_rate_real_top20"], errors="coerce").mean()),
        "avg_turnover": float(pd.to_numeric(g["buy_turnover"], errors="coerce").mean()),
        "avg_trade_cost": float(pd.to_numeric(g["trade_cost"], errors="coerce").mean()),
        "avg_board_hhi": float(pd.to_numeric(g["board_hhi"], errors="coerce").mean()),
        "avg_chinext_ratio": float(pd.to_numeric(g["board_chinext_ratio"], errors="coerce").mean()),
        "avg_star_ratio": float(pd.to_numeric(g["board_star_ratio"], errors="coerce").mean()),
        "robust_pass_proxy": robust_pass,
    })


method_summary_df = rolling_monthly_df.groupby("train_policy").apply(lambda g: summarize_policy_monthly(g, "proxy_net_excess_ret")).reset_index(drop=True)
method_summary_df = method_summary_df.sort_values(["robust_pass_proxy", "cum_ret"], ascending=[False, False])

common_core_df = rolling_monthly_df[(rolling_monthly_df["train_policy"].isin(CORE_COMMON_POLICIES)) & (rolling_monthly_df[DATE_COL] >= common_core_start)].copy()
common_core_summary_df = common_core_df.groupby("train_policy").apply(lambda g: summarize_policy_monthly(g, "proxy_net_excess_ret")).reset_index(drop=True) if not common_core_df.empty else pd.DataFrame()
if not common_core_summary_df.empty:
    common_core_summary_df["common_window"] = "core_" + pd.Timestamp(common_core_start).strftime("%Y%m")
    common_core_summary_df = common_core_summary_df.sort_values("cum_ret", ascending=False)

common_all_df = rolling_monthly_df[rolling_monthly_df[DATE_COL] >= common_all_start].copy()
common_all_summary_df = common_all_df.groupby("train_policy").apply(lambda g: summarize_policy_monthly(g, "proxy_net_excess_ret")).reset_index(drop=True) if not common_all_df.empty else pd.DataFrame()
if not common_all_summary_df.empty:
    common_all_summary_df["common_window"] = "all_" + pd.Timestamp(common_all_start).strftime("%Y%m")
    common_all_summary_df = common_all_summary_df.sort_values("cum_ret", ascending=False)

fold_summary_rows = []
for (train_policy, fold_id), g in rolling_monthly_df.groupby(["train_policy", "fold_id"]):
    st = summarize_return_series(g.sort_values(DATE_COL)["proxy_net_excess_ret"])
    fold_summary_rows.append({
        "train_policy": train_policy,
        "fold_id": fold_id,
        "train_start": g["train_start"].iloc[0],
        "train_end": g["train_end"].iloc[0],
        "test_start": g["test_start"].iloc[0],
        "test_end": g["test_end"].iloc[0],
        "months": st["months"],
        "fold_cum_ret": st["cum_ret"],
        "fold_mean_ret": st["mean_ret"],
        "fold_win_rate": st["win_rate"],
        "fold_max_drawdown": st["max_drawdown"],
        "rank_ic_mean": float(g["rank_ic"].mean()),
        "random_alpha_percentile_mean": float(g["random_alpha_percentile"].mean()),
        "avg_board_hhi": float(g["board_hhi"].mean()),
        "avg_chinext_ratio": float(g["board_chinext_ratio"].mean()),
        "avg_star_ratio": float(g["board_star_ratio"].mean()),
    })
fold_summary_df = pd.DataFrame(fold_summary_rows).sort_values(["train_policy", "test_start"])

yearly_rows = []
ytmp = rolling_monthly_df.copy()
ytmp["year"] = pd.to_datetime(ytmp[DATE_COL]).dt.year
for (train_policy, year), g in ytmp.groupby(["train_policy", "year"]):
    st = summarize_return_series(g.sort_values(DATE_COL)["proxy_net_excess_ret"])
    yearly_rows.append({
        "train_policy": train_policy,
        "year": int(year),
        "months": st["months"],
        "cum_ret": st["cum_ret"],
        "mean_ret": st["mean_ret"],
        "win_rate": st["win_rate"],
        "max_drawdown": st["max_drawdown"],
        "rank_ic_mean": float(g["rank_ic"].mean()),
        "avg_random_alpha_percentile": float(g["random_alpha_percentile"].mean()),
        "p25_random_alpha_percentile": float(g["random_alpha_percentile"].quantile(0.25)),
    })
yearly_summary_df = pd.DataFrame(yearly_rows).sort_values(["train_policy", "year"])

display(method_summary_df)
display(common_core_summary_df)
display(common_all_summary_df)
display(fold_summary_df.head(30))
display(yearly_summary_df)


In [ ]:
# Same-cutoff comparison: this is the main table for separating train-length effect from regime/cutoff effect.
length_rows = []
for test_start, g in rolling_monthly_df.groupby("test_start"):
    for train_policy, pg in g.groupby("train_policy"):
        st = summarize_return_series(pg.sort_values(DATE_COL)["proxy_net_excess_ret"])
        length_rows.append({
            "test_start": test_start,
            "test_end": pg["test_end"].iloc[0],
            "train_policy": train_policy,
            "method_type": pg["method_type"].iloc[0],
            "train_start": pg["train_start"].iloc[0],
            "train_end": pg["train_end"].iloc[0],
            "train_window_months": pg["train_window_months"].dropna().iloc[0] if pg["train_window_months"].notnull().any() else np.nan,
            "train_months": pd.to_datetime(pg["train_end"].iloc[0]).to_period("M").ordinal - pd.to_datetime(pg["train_start"].iloc[0]).to_period("M").ordinal + 1,
            "oos_months": st["months"],
            "oos_cum_ret": st["cum_ret"],
            "oos_mean_ret": st["mean_ret"],
            "oos_win_rate": st["win_rate"],
            "oos_max_drawdown": st["max_drawdown"],
            "rank_ic_mean": float(pg["rank_ic"].mean()),
            "avg_random_alpha_percentile": float(pg["random_alpha_percentile"].mean()),
        })
length_comparison_df = pd.DataFrame(length_rows).sort_values(["test_start", "train_policy"])

best_by_cutoff_rows = []
for test_start, g in length_comparison_df.groupby("test_start"):
    gg = g.dropna(subset=["oos_cum_ret"]).copy()
    if gg.empty:
        continue
    best = gg.sort_values("oos_cum_ret", ascending=False).iloc[0]
    best_by_cutoff_rows.append({
        "test_start": test_start,
        "available_methods": int(len(gg)),
        "best_train_policy": best["train_policy"],
        "best_oos_cum_ret": best["oos_cum_ret"],
        "median_oos_cum_ret": float(gg["oos_cum_ret"].median()),
        "worst_oos_cum_ret": float(gg["oos_cum_ret"].min()),
        "dispersion_best_minus_worst": float(gg["oos_cum_ret"].max() - gg["oos_cum_ret"].min()),
    })
best_by_cutoff_df = pd.DataFrame(best_by_cutoff_rows).sort_values("test_start")

failure_months_df = rolling_monthly_df[
    (rolling_monthly_df["proxy_net_excess_ret"] < 0)
    & ((rolling_monthly_df["random_alpha_percentile"] < 0.20) | (rolling_monthly_df["rank_ic"] < 0))
].copy()
failure_months_df = failure_months_df.sort_values(["proxy_net_excess_ret", "random_alpha_percentile"])

latest_dt = rolling_monthly_df[DATE_COL].max()
latest_targets_df = rolling_monthly_df[rolling_monthly_df[DATE_COL] == latest_dt].copy()

print("length comparison sample")
display(length_comparison_df.tail(40))
print("best by cutoff")
display(best_by_cutoff_df)
print("failure months")
display(failure_months_df[["train_policy", DATE_COL, "fold_id", "proxy_net_excess_ret", "random_alpha_percentile", "rank_ic", "board_hhi", "targets"]].head(40))
print("latest targets")
display(latest_targets_df[["train_policy", DATE_COL, "proxy_net_excess_ret", "random_alpha_percentile", "rank_ic", "board_hhi", "targets"]])


## 可选：JQ-like 日频账户模拟

这段和 V61 的目的相同：把每月 target 串成更接近聚宽回测的账户路径。

- 调仓日卖出不在新 target 的持仓；
- 保留重叠持仓；
- 只用现金买入新增 target；
- 买入按开盘价，净值按收盘价；
- 考虑滑点、佣金、印花税和整手约束；
- 只在能调用 JoinQuant `get_price` 的环境里自动运行。

如果本地普通 Python 环境没有 `get_price`，会自动跳过，不影响 proxy OOS 结果。


In [ ]:
def parse_target_list(x):
    if pd.isnull(x):
        return []
    return [s.strip() for s in str(x).split(",") if s.strip()]


def is_kcb_stock(stock):
    return str(stock).startswith(("688", "689"))


def lot_size_for_stock(stock):
    return KCB_MIN_LOT if is_kcb_stock(stock) else NORMAL_MIN_LOT


def floor_to_lot(value, price, lot):
    if pd.isnull(price) or price <= 0 or value <= 0:
        return 0
    return int(np.floor(value / price / lot) * lot)


def calc_buy_cash_cost(amount, price):
    gross = float(amount) * float(price)
    commission = max(MIN_COMMISSION, gross * OPEN_COMMISSION)
    slippage = gross * SLIPPAGE_RATE
    return gross + commission + slippage


def calc_sell_cash_in(amount, price):
    gross = float(amount) * float(price)
    commission = max(MIN_COMMISSION, gross * CLOSE_COMMISSION)
    tax = gross * CLOSE_TAX
    slippage = gross * SLIPPAGE_RATE
    return gross - commission - tax - slippage


def jq_fetch_daily_price_panel(securities, start_date, end_date, fields, chunk_size=120):
    securities = unique_keep_order([s for s in securities if isinstance(s, str) and s])
    if len(securities) == 0:
        return pd.DataFrame()
    parts = []
    chunks_list = list(chunks(securities, chunk_size))
    for sec_chunk in progress_iter(chunks_list, total=len(chunks_list), desc="jq price chunks", leave=False):
        try:
            try:
                df = get_price(
                    sec_chunk,
                    start_date=pd.Timestamp(start_date).strftime("%Y-%m-%d"),
                    end_date=pd.Timestamp(end_date).strftime("%Y-%m-%d"),
                    frequency="daily",
                    fields=fields,
                    skip_paused=False,
                    fq=JQ_SIM_PRICE_FQ,
                    panel=False,
                    fill_paused=True,
                )
            except TypeError:
                df = get_price(
                    sec_chunk,
                    start_date=pd.Timestamp(start_date).strftime("%Y-%m-%d"),
                    end_date=pd.Timestamp(end_date).strftime("%Y-%m-%d"),
                    frequency="daily",
                    fields=fields,
                    skip_paused=False,
                    fq=JQ_SIM_PRICE_FQ,
                    panel=False,
                )
        except NameError:
            raise RuntimeError("JoinQuant get_price is not available in this environment")
        if df is not None and not df.empty:
            parts.append(df.copy())
    if not parts:
        return pd.DataFrame()
    out = pd.concat(parts, axis=0, ignore_index=True, sort=False)
    if "time" in out.columns:
        out["time"] = pd.to_datetime(out["time"]).dt.normalize()
    return out


def jq_fetch_benchmark_close(start_date, end_date):
    try:
        try:
            df = get_price(
                BENCHMARK,
                start_date=pd.Timestamp(start_date).strftime("%Y-%m-%d"),
                end_date=pd.Timestamp(end_date).strftime("%Y-%m-%d"),
                frequency="daily",
                fields=[JQ_SIM_MARK_FIELD],
                skip_paused=False,
                fq=JQ_SIM_PRICE_FQ,
            )
        except TypeError:
            df = get_price(
                BENCHMARK,
                start_date=pd.Timestamp(start_date).strftime("%Y-%m-%d"),
                end_date=pd.Timestamp(end_date).strftime("%Y-%m-%d"),
                frequency="daily",
                fields=[JQ_SIM_MARK_FIELD],
                skip_paused=False,
                fq=JQ_SIM_PRICE_FQ,
            )
    except NameError:
        raise RuntimeError("JoinQuant get_price is not available in this environment")
    if df is None or len(df) == 0:
        return pd.Series(dtype=float)
    if isinstance(df, pd.Series):
        s = df.copy()
    elif JQ_SIM_MARK_FIELD in df.columns:
        s = df[JQ_SIM_MARK_FIELD].copy()
    else:
        s = df.iloc[:, 0].copy()
    s.index = pd.to_datetime(s.index).normalize()
    return pd.to_numeric(s, errors="coerce").dropna()


def price_matrix(price_df, field):
    if price_df is None or price_df.empty or "time" not in price_df.columns or "code" not in price_df.columns or field not in price_df.columns:
        return pd.DataFrame()
    return price_df.pivot_table(index="time", columns="code", values=field).sort_index()


def get_price_from_row(mat, dt, stock, fallback=np.nan):
    try:
        v = mat.at[pd.Timestamp(dt).normalize(), stock]
    except Exception:
        v = fallback
    if pd.isnull(v):
        return fallback
    return float(v)


def simulate_one_policy_jq_like(policy_monthly_df):
    gdf = policy_monthly_df.sort_values(DATE_COL).copy()
    if gdf.empty:
        return pd.DataFrame(), pd.DataFrame(), {"jq_like_status": "empty"}
    target_by_date = {}
    all_stocks = []
    for _, row in gdf.iterrows():
        dt = pd.Timestamp(row[DATE_COL]).normalize()
        stocks = parse_target_list(row.get("targets", ""))
        target_by_date[dt] = stocks
        all_stocks.extend(stocks)
    all_stocks = unique_keep_order(all_stocks)
    start_date = pd.Timestamp(gdf[DATE_COL].min()).normalize()
    if "next_date" in gdf.columns and gdf["next_date"].notnull().any():
        end_date = pd.Timestamp(gdf["next_date"].dropna().max()).normalize()
    else:
        end_date = pd.Timestamp(gdf[DATE_COL].max()).normalize()
    if len(all_stocks) == 0:
        return pd.DataFrame(), pd.DataFrame(), {"jq_like_status": "no_targets"}

    price_df = jq_fetch_daily_price_panel(
        all_stocks,
        start_date,
        end_date,
        [JQ_SIM_BUY_TIME_FIELD, JQ_SIM_MARK_FIELD],
        chunk_size=JQ_SIM_CHUNK_SIZE,
    )
    open_mat = price_matrix(price_df, JQ_SIM_BUY_TIME_FIELD)
    close_mat = price_matrix(price_df, JQ_SIM_MARK_FIELD)
    if close_mat.empty:
        return pd.DataFrame(), pd.DataFrame(), {"jq_like_status": "no_price"}
    if open_mat.empty:
        open_mat = close_mat.copy()
    else:
        open_mat = open_mat.combine_first(close_mat)

    bench_close = jq_fetch_benchmark_close(start_date, end_date)
    trade_dates = list(close_mat.index)
    cash = float(JQ_SIM_INITIAL_CASH)
    positions = {}
    last_mark = {}
    daily_rows = []
    trade_rows = []
    target_dates = set(target_by_date.keys())

    for cur_date in progress_iter(trade_dates, total=len(trade_dates), desc="jq days", leave=False):
        if cur_date in target_dates:
            targets = target_by_date.get(cur_date, [])
            target_set = set(targets)
            for stock in list(positions.keys()):
                amount = positions.get(stock, 0)
                if amount <= 0 or stock in target_set:
                    continue
                px = get_price_from_row(open_mat, cur_date, stock, last_mark.get(stock, np.nan))
                if pd.isnull(px) or px <= 0:
                    continue
                cash_in = calc_sell_cash_in(amount, px)
                cash += cash_in
                trade_rows.append({"date": cur_date, "stock": stock, "side": "sell", "amount": amount, "price": px, "cash_delta": cash_in})
                positions.pop(stock, None)

            current_holds = [s for s, amount in positions.items() if amount > 0]
            target_num = len(targets)
            buy_list = [s for s in targets if positions.get(s, 0) <= 0]
            buy_slots = max(0, target_num - len(current_holds))
            if buy_slots > 0 and cash > 0:
                value_per_slot = cash / float(buy_slots)
                for stock in buy_list:
                    px = get_price_from_row(open_mat, cur_date, stock, get_price_from_row(close_mat, cur_date, stock))
                    if pd.isnull(px) or px <= 0:
                        continue
                    lot = lot_size_for_stock(stock)
                    amount = floor_to_lot(value_per_slot, px * (1.0 + BUY_COST_RATE), lot)
                    if amount < lot:
                        continue
                    cash_cost = calc_buy_cash_cost(amount, px)
                    while amount >= lot and cash_cost > cash:
                        amount -= lot
                        cash_cost = calc_buy_cash_cost(amount, px) if amount > 0 else 0.0
                    if amount < lot or cash_cost <= 0:
                        continue
                    cash -= cash_cost
                    positions[stock] = positions.get(stock, 0) + amount
                    trade_rows.append({"date": cur_date, "stock": stock, "side": "buy", "amount": amount, "price": px, "cash_delta": -cash_cost})

        pos_value = 0.0
        valid_marks = 0
        for stock, amount in list(positions.items()):
            px = get_price_from_row(close_mat, cur_date, stock, last_mark.get(stock, np.nan))
            if pd.isnull(px) or px <= 0:
                continue
            last_mark[stock] = px
            pos_value += float(amount) * float(px)
            valid_marks += 1
        total_value = cash + pos_value
        daily_rows.append({
            "date": cur_date,
            "cash": cash,
            "position_value": pos_value,
            "total_value": total_value,
            "position_count": int(len([s for s, a in positions.items() if a > 0])),
            "valid_marks": int(valid_marks),
            "is_rebalance_date": bool(cur_date in target_dates),
        })

    daily_df = pd.DataFrame(daily_rows).sort_values("date")
    if daily_df.empty:
        return pd.DataFrame(), pd.DataFrame(), {"jq_like_status": "empty_daily"}
    daily_df["nav"] = daily_df["total_value"] / float(JQ_SIM_INITIAL_CASH)
    daily_df["drawdown"] = daily_df["nav"] / daily_df["nav"].cummax() - 1.0
    if len(bench_close) > 1:
        bench = bench_close.reindex(daily_df["date"]).ffill().dropna()
        if len(bench) > 1 and bench.iloc[0] > 0:
            bench_nav = bench / bench.iloc[0]
            daily_df = daily_df.merge(pd.DataFrame({"date": bench_nav.index, "benchmark_nav": bench_nav.values}), on="date", how="left")
            daily_df["benchmark_nav"] = daily_df["benchmark_nav"].ffill()
            daily_df["relative_excess_nav"] = daily_df["nav"] / daily_df["benchmark_nav"]
    if "benchmark_nav" not in daily_df.columns:
        daily_df["benchmark_nav"] = np.nan
        daily_df["relative_excess_nav"] = np.nan

    month_rows = []
    prev_value = float(JQ_SIM_INITIAL_CASH)
    prev_bench_nav = 1.0
    for _, row in gdf.iterrows():
        end_dt = pd.Timestamp(row["next_date"]).normalize() if "next_date" in row and not pd.isnull(row.get("next_date")) else pd.Timestamp(row[DATE_COL]).normalize()
        eligible = daily_df[daily_df["date"] <= end_dt]
        if eligible.empty:
            continue
        end_row = eligible.iloc[-1]
        period_ret = float(end_row["total_value"] / prev_value - 1.0) if prev_value > 0 else np.nan
        bench_nav = float(end_row.get("benchmark_nav", np.nan))
        bench_ret = bench_nav / prev_bench_nav - 1.0 if not pd.isnull(bench_nav) and prev_bench_nav > 0 else np.nan
        excess_ret = (1.0 + period_ret) / (1.0 + bench_ret) - 1.0 if not pd.isnull(period_ret) and not pd.isnull(bench_ret) else np.nan
        out = dict(row)
        out.update({
            "jq_like_period_end": end_row["date"],
            "jq_like_total_value": float(end_row["total_value"]),
            "jq_like_nav": float(end_row["nav"]),
            "jq_like_period_ret": period_ret,
            "jq_like_benchmark_period_ret": bench_ret,
            "jq_like_excess_period_ret": excess_ret,
            "jq_like_drawdown": float(end_row["drawdown"]),
            "jq_like_position_count": int(end_row["position_count"]),
            "jq_like_cash": float(end_row["cash"]),
        })
        month_rows.append(out)
        prev_value = float(end_row["total_value"])
        if not pd.isnull(bench_nav):
            prev_bench_nav = bench_nav

    status = {
        "jq_like_status": "ok",
        "jq_like_days": int(len(daily_df)),
        "jq_like_trades": int(len(trade_rows)),
        "jq_like_final_value": float(daily_df["total_value"].iloc[-1]),
        "jq_like_cum_ret": float(daily_df["nav"].iloc[-1] - 1.0),
        "jq_like_max_drawdown": float(daily_df["drawdown"].min()),
    }
    if daily_df["benchmark_nav"].notnull().any():
        status["jq_like_benchmark_cum_ret"] = float(daily_df["benchmark_nav"].dropna().iloc[-1] - 1.0)
        status["jq_like_relative_excess_cum_ret"] = float(daily_df["relative_excess_nav"].dropna().iloc[-1] - 1.0)
    else:
        status["jq_like_benchmark_cum_ret"] = np.nan
        status["jq_like_relative_excess_cum_ret"] = np.nan
    return daily_df, pd.DataFrame(month_rows), status


def maybe_run_jq_like_simulator(eval_monthly_df):
    if RUN_JQ_LIKE_SIMULATOR == "OFF" or eval_monthly_df.empty:
        print("JQ-like simulator disabled or no monthly rows")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    daily_parts = []
    monthly_parts = []
    status_rows = []
    groups = list(eval_monthly_df.groupby("train_policy"))
    for train_policy, gdf in progress_iter(groups, total=len(groups), desc="jq-like policies"):
        try:
            daily_one, monthly_one, status = simulate_one_policy_jq_like(gdf)
            status.update({"train_policy": train_policy})
            if not daily_one.empty:
                daily_one = daily_one.copy()
                daily_one["train_policy"] = train_policy
                daily_parts.append(daily_one)
            if not monthly_one.empty:
                monthly_one = monthly_one.copy()
                monthly_one["jq_like_available"] = True
                monthly_parts.append(monthly_one)
            status_rows.append(status)
        except RuntimeError as err:
            if RUN_JQ_LIKE_SIMULATOR == "AUTO":
                print("JQ-like simulator skipped:", err)
                return pd.DataFrame(), pd.DataFrame(), pd.DataFrame([{"jq_like_status": str(err)}])
            raise
        except Exception as err:
            status_rows.append({"train_policy": train_policy, "jq_like_status": str(err)})
    daily_df = pd.concat(daily_parts, ignore_index=True, sort=False) if daily_parts else pd.DataFrame()
    monthly_df = pd.concat(monthly_parts, ignore_index=True, sort=False) if monthly_parts else pd.DataFrame()
    status_df = pd.DataFrame(status_rows)
    print("JQ-like daily/monthly/status:", daily_df.shape, monthly_df.shape, status_df.shape)
    return daily_df, monthly_df, status_df


In [ ]:
jq_like_daily_equity_df, jq_like_monthly_df, jq_like_status_df = maybe_run_jq_like_simulator(rolling_monthly_df)
if not jq_like_status_df.empty:
    display(jq_like_status_df)

jq_like_summary_df = pd.DataFrame()
if not jq_like_monthly_df.empty:
    jq_like_summary_df = jq_like_monthly_df.groupby("train_policy").apply(lambda g: summarize_policy_monthly(g, "jq_like_excess_period_ret")).reset_index(drop=True)
    jq_like_summary_df = jq_like_summary_df.rename(columns={
        "cum_ret": "jq_like_excess_cum_ret",
        "ann_ret": "jq_like_excess_ann_ret",
        "mean_monthly_ret": "jq_like_excess_mean_monthly_ret",
        "win_rate": "jq_like_excess_win_rate",
        "max_drawdown": "jq_like_excess_max_drawdown",
        "sharpe12": "jq_like_excess_sharpe12",
        "robust_pass_proxy": "robust_pass_jq_like",
    })
    display(jq_like_summary_df)


In [ ]:
# Save all outputs.
rolling_monthly_df.to_csv(OUT_DIR / "v65_rolling_oos_monthly.csv", index=False)
fold_plan_df.to_csv(OUT_DIR / "v65_fold_plan.csv", index=False)
fold_meta_df.to_csv(OUT_DIR / "v65_fold_meta.csv", index=False)
score_panel_df.to_csv(OUT_DIR / "v65_score_panel.csv", index=False)
method_summary_df.to_csv(OUT_DIR / "v65_method_summary_proxy.csv", index=False)
common_core_summary_df.to_csv(OUT_DIR / "v65_common_core_summary_proxy.csv", index=False)
common_all_summary_df.to_csv(OUT_DIR / "v65_common_all_summary_proxy.csv", index=False)
fold_summary_df.to_csv(OUT_DIR / "v65_fold_summary_proxy.csv", index=False)
yearly_summary_df.to_csv(OUT_DIR / "v65_yearly_summary_proxy.csv", index=False)
length_comparison_df.to_csv(OUT_DIR / "v65_train_length_same_cutoff_comparison.csv", index=False)
best_by_cutoff_df.to_csv(OUT_DIR / "v65_best_method_by_cutoff.csv", index=False)
failure_months_df.to_csv(OUT_DIR / "v65_failure_months.csv", index=False)
latest_targets_df.to_csv(OUT_DIR / "v65_latest_targets_by_method.csv", index=False)
jq_like_daily_equity_df.to_csv(OUT_DIR / "v65_jq_like_daily_equity.csv", index=False)
jq_like_monthly_df.to_csv(OUT_DIR / "v65_jq_like_monthly.csv", index=False)
jq_like_status_df.to_csv(OUT_DIR / "v65_jq_like_status.csv", index=False)
jq_like_summary_df.to_csv(OUT_DIR / "v65_method_summary_jq_like.csv", index=False)

print("saved outputs:")
for p in sorted(OUT_DIR.glob("v65_*.csv")):
    print("-", p)


## 怎么读 V65

优先顺序：

1. `v65_method_summary_proxy.csv`：每种滚动训练方法各跑各的完整 OOS 表现。
2. `v65_common_core_summary_proxy.csv`：expanding / rolling36 / rolling48 / rolling60 在同一 OOS 起点后的公平比较。
3. `v65_common_all_summary_proxy.csv`：包含 rolling72 的共同区间比较，但样本更短。
4. `v65_train_length_same_cutoff_comparison.csv`：同一个 cutoff 下，不同训练长度谁更好，用来回答“数据量问题”。
5. `v65_best_method_by_cutoff.csv`：看最优训练长度是否随 cutoff/regime 变化。
6. `v65_yearly_summary_proxy.csv` 和 `v65_failure_months.csv`：看是否 2023 / 2025-03 / 2025-05 仍然是统一失效期。
7. 如果聚宽环境可用，最终看 `v65_method_summary_jq_like.csv` 和 `v65_jq_like_daily_equity.csv`。

解释原则：

- 如果 expanding 稳定优于 rolling：长期样本/数据量更重要。
- 如果 rolling60/48 稳定优于 expanding：旧样本衰减，近年 regime 更重要。
- 如果同一 cutoff 下所有训练长度都差：不是数据量问题，而是方法论在该市场状态失效。
- 如果某些 cutoff 之后所有方法都变好：训练集中加入了关键 regime 样本，说明 cutoff/regime 效应强。
- 如果 proxy 通过但 JQ-like 不通过：组合收益被交易/现金/整手约束吃掉，不能进主线。
- 如果 proxy 和 JQ-like 都通过：这套滚动训练方法才算真正接近“可生产”。
